# BERT/GPT/Transformer 中文情感训练
本质上就是迁移学习
1. backbone 特征提取(上游模型)
2. 将特征提取结果作为神经网络模型的输入输出结果(下游模型) 可以重新训练

## BERT
Bert(Bidirectional Encoder Representations from Transformers) 是一种由Google开发的预训练模型。 传统的模型只能从左到右理解上下文，而Bert能够看到单词左右两边的内容，更准确的理解语义。
### BERT特点
1. 双向编码器
    - 双向理解上下文
    - BERT能够同时看到单词左右两边
2. Transformer架构
    - 基于Transformer的 Encoder 部分
    - 使用自注意力机制(Self-Attention) 捕捉词与词之间的关系

In [32]:

from datasets import load_dataset
from transformers import BertTokenizer


In [33]:
import os

os.environ['http_proxy'] = '127.0.0.1:10809'
os.environ['https_proxy'] = '127.0.0.1:10809'

## 加载预训练模型的Tokenizer

In [34]:
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese', cache_dir='./.cache')
tokenizer

BertTokenizer(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [35]:
from transformers.utils import PaddingStrategy
from transformers import TensorType
from transformers.tokenization_utils_base import TruncationStrategy

out = tokenizer._encode_plus(
    text=['从明天起，做一个信服的人。', '喂马，劈柴，周游世界。'],
    truncation_strategy=TruncationStrategy.LONGEST_FIRST,
    padding_strategy=PaddingStrategy.MAX_LENGTH,
    max_length=17,
    return_tensors=TensorType.PYTORCH,
    return_length=True
)

In [36]:
for k, v in out.items():
    print(k, v)

input_ids tensor([[ 101,  794, 3209, 1921, 6629, 8024,  976,  671,  702,  928, 3302, 4638,
          782,  511,  102,    0,    0],
        [ 101, 1585, 7716, 8024, 1207, 3395, 8024, 1453, 3952,  686, 4518,  511,
          102,    0,    0,    0,    0]])
token_type_ids tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])
length tensor([17, 17])


In [37]:
 print(tokenizer.decode(out['input_ids'][0]))

[CLS] 从 明 天 起 ， 做 一 个 信 服 的 人 。 [SEP] [PAD] [PAD]


## 定义数据集

In [38]:
# 下载数据集 并保存到本地用于后续 从本地加载数据集 ! 这一步只是用于练习
# dataset = load_dataset('lansinuote/ChnSentiCorp', cache_dir='./.cache')
# dataset.save_to_disk('./.cache/ChnSentiCorp')

In [39]:
import torch
from datasets import load_from_disk

In [40]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, key):
        self.dataset = load_from_disk('./.cache/ChnSentiCorp')[key]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]['text']
        label = self.dataset[idx]['label']
        return text, label

In [41]:
dataset = Dataset('train')
len(dataset)

9600

In [42]:
dataset[20]

('非常不错，服务很好，位于市中心区，交通方便，不过价格也高！', 1)

## 定义计算设备

In [43]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import torch_directml

device = torch_directml.device() if device == 'cpu' else device
device

device(type='privateuseone', index=0)

## 数据整理
从我们的tokenizer中获取的输入数数据如下
```text
input_ids tensor([[ 101,  794, 3209, 1921, 6629, 8024,  976,  671,  702,  928, 3302, 4638,
          782,  511,  102,    0,    0],
        [ 101, 1585, 7716, 8024, 1207, 3395, 8024, 1453, 3952,  686, 4518,  511,
          102,    0,    0,    0,    0]])
token_type_ids tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])
length tensor([17, 17])
```
看出我们输入有三个信息 input_ids, token_type_ids, attention_mask

In [44]:
def collate_fn(data):
    text = [i[0] for i in data]
    labels = [i[1] for i in data]

    # 编码
    data = tokenizer.__call__(
        text=text,
        truncation=True,
        padding=PaddingStrategy.MAX_LENGTH,
        max_length=500,
        return_tensors=TensorType.PYTORCH,
        return_length=True,
    )
    # input_ids 编码之后的数字
    # attention_mask 0的位置是不需要计算attention的 1的位置表示需要计算attention
    # token_type_ids token的类型 - 表示第一个句子 1表示第二个句子
    input_ids = data['input_ids']
    attention_mask = data['attention_mask']
    token_type_ids = data['token_type_ids']
    labels = torch.LongTensor(labels)

    # 拷贝到设备
    if device != 'cpu':
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        token_type_ids = token_type_ids.to(device)
    return input_ids, attention_mask, token_type_ids, labels

In [45]:
# 测试
data = [
    ['从明天起，做一个信服的人。', 1],
    ['喂马，劈柴，周游世界。', 1],
    ['我太菜了，下次要加油。', 0],
]
collate_fn(data)

(tensor([[ 101,  794, 3209,  ...,    0,    0,    0],
         [ 101, 1585, 7716,  ...,    0,    0,    0],
         [ 101, 2769, 1922,  ...,    0,    0,    0]], device='privateuseone:0'),
 tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]], device='privateuseone:0'),
 tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]], device='privateuseone:0'),
 tensor([1, 1, 0]))

In [46]:
# 数据加载器
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=16,
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True
)
data_loader

In [47]:
len(data_loader)

600

In [48]:
# 查看数据样例
case_data = None
for i, case_data in enumerate(data_loader):
    break
[x.shape for x in case_data]

[torch.Size([16, 500]),
 torch.Size([16, 500]),
 torch.Size([16, 500]),
 torch.Size([16])]

## 加载预训练模型

In [49]:
from transformers import BertModel

In [50]:
pretrained_model = BertModel.from_pretrained('bert-base-chinese', cache_dir='./.cache')
pretrained_model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4861.93it/s]
BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(21128, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [51]:
# 计算参数量
sum([p.numel() for p in pretrained_model.parameters()])

102267648

## 迁移学习
1. 冻结参数
2. 重构和重新训练下游游神经网络

In [52]:
# 冻结参数
for p in pretrained_model.parameters():
    p.requires_grad_(False)

In [53]:
if device != 'cpu':
    pretrained_model.to(device)

In [54]:
# 试算 生成特征
out = pretrained_model(input_ids=case_data[0], attention_mask=case_data[1], token_type_ids=case_data[2])
out

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 0.5457,  0.6655,  0.2250,  ..., -0.7520, -0.1046, -0.0579],
         [-0.2764,  0.2270,  0.5576,  ..., -1.3194, -0.2891, -0.0802],
         [ 0.9597, -0.4227, -0.3807,  ..., -0.1193, -0.3370, -0.0719],
         ...,
         [-0.1875, -0.0519, -0.1040,  ...,  0.2691, -0.0082, -0.4384],
         [ 0.2220, -0.1958, -0.5815,  ...,  0.3013, -0.1420, -0.3296],
         [ 0.1460, -0.0400, -0.1251,  ..., -0.0173, -0.2592, -0.3783]],

        [[ 0.2865, -0.2421,  0.6379,  ..., -0.9725, -0.0534,  0.0066],
         [ 0.1127,  0.6956,  0.1313,  ..., -0.9257, -0.6642,  0.3335],
         [ 0.9165, -0.0974,  0.3406,  ...,  1.1473,  0.7430,  0.3622],
         ...,
         [ 0.3390,  0.3353,  0.2556,  ...,  0.0529, -0.1400, -0.4945],
         [ 0.1416,  0.3060,  0.1458,  ...,  0.1152, -0.1473, -0.5301],
         [ 0.4719,  0.2204, -0.2037,  ...,  0.0852, -0.0160, -0.2276]],

        [[-0.4125, -0.1536, -0.8868,  ..., -0.5406,  

In [56]:
# 查看最后一层输出的维度 为768维
# [batch_size, seq_len, hidden]
out.last_hidden_state.shape

torch.Size([16, 500, 768])

In [57]:
# 定义下游任务模型
class Model(torch.nn.Module):
    def __init__(self, pretrained_model):
        super().__init__()
        # 将上游输出的768维映射为2维 分类任务
        self.fc = torch.nn.Linear(768, 2)
        self.pretrained_model = pretrained_model

    def forward(self, input_ids, attention_mask, token_type_ids):
        # 使用上游模型获取特征
        with torch.no_grad():
            out = self.pretrained_model(input_ids=input_ids, attention_mask=attention_mask,
                                        token_type_ids=token_type_ids)
        # 在 BERT 中，每个输入序列的开头都会自动添加一个特殊标记 [CLS]（Classification Token）。这个 [CLS] 标记的位置（索引 0）的隐藏状态被设计用来表示整个句子的聚合表示。
        out = self.fc(out.last_hidden_state[:, 0, :])
        out = out.softmax(dim=1)
        return out

In [58]:
model = Model(pretrained_model)

In [59]:
# 试算
model(input_ids=case_data[0], attention_mask=case_data[1], token_type_ids=case_data[2])

tensor([[0.5062, 0.4938],
        [0.4906, 0.5094],
        [0.5888, 0.4112],
        [0.5419, 0.4581],
        [0.5145, 0.4855],
        [0.4876, 0.5124],
        [0.5430, 0.4570],
        [0.6309, 0.3691],
        [0.4261, 0.5739],
        [0.3846, 0.6154],
        [0.5485, 0.4515],
        [0.5542, 0.4458],
        [0.5877, 0.4123],
        [0.4476, 0.5524],
        [0.5619, 0.4381],
        [0.5252, 0.4748]], device='privateuseone:0',
       grad_fn=<SoftmaxBackward0>)

## 训练

In [60]:
from transformers.optimization import get_scheduler
from torch.optim import AdamW

In [67]:
def train(model):
    # 定义优化器
    optimizer = AdamW(model.parameters(), lr=5e-4)
    # 定义损失函数
    criterion = torch.nn.CrossEntropyLoss()
    # 定义学习率调节器
    scheduler = get_scheduler(
        name='linear',
        num_warmup_steps=0,
        num_training_steps=len(data_loader),
        optimizer=optimizer
    )
    # 切换到训练模式
    model.train()

    # 按批次遍历训练集中的数据
    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(data_loader):
        # 模型计算
        if device != 'cpu':
            input_ids = input_ids = input_ids.to(device)
            attention_mask = attention_mask = attention_mask.to(device)
            token_type_ids = token_type_ids = token_type_ids.to(device)
            labels = labels.to(device)
        out = model(input_ids=input_ids, attention_mask=attention_mask,
                    token_type_ids=token_type_ids)
        # 计算损失
        loss = criterion(out, labels)
        # 反向传播
        loss.backward()
        optimizer.step()
        scheduler.step()
        # 清空梯度
        optimizer.zero_grad()

        if i % 10 == 0:
            out = out.argmax(dim=1)
            accuracy = (out == labels).sum().item() / len(labels)
            lr = optimizer.state_dict()['param_groups'][0]['lr']
            print(f'batch: {i}, loss: {loss.item():.4f}, accuracy: {accuracy:.9f}, lr: {lr:.4f}')

In [68]:
model = Model(pretrained_model)
model.to(device)
train(model)

D:\Work\transformers-learn-demo\.venv1\lib\site-packages\torch\optim\adamw.py:529: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


batch: 0, loss: 0.7259, accuracy: 0.5000, lr: 0.0005
batch: 10, loss: 0.7209, accuracy: 0.3750, lr: 0.0005
batch: 20, loss: 0.5859, accuracy: 0.8750, lr: 0.0005
batch: 30, loss: 0.5530, accuracy: 0.8750, lr: 0.0005
batch: 40, loss: 0.5171, accuracy: 0.9375, lr: 0.0005
batch: 50, loss: 0.5064, accuracy: 0.8750, lr: 0.0005
batch: 60, loss: 0.5138, accuracy: 0.9375, lr: 0.0004
batch: 70, loss: 0.5485, accuracy: 0.8750, lr: 0.0004
batch: 80, loss: 0.5875, accuracy: 0.7500, lr: 0.0004
batch: 90, loss: 0.5290, accuracy: 0.8750, lr: 0.0004
batch: 100, loss: 0.4943, accuracy: 0.9375, lr: 0.0004
batch: 110, loss: 0.5067, accuracy: 0.8750, lr: 0.0004
batch: 120, loss: 0.5161, accuracy: 0.8125, lr: 0.0004
batch: 130, loss: 0.4733, accuracy: 0.9375, lr: 0.0004
batch: 140, loss: 0.5414, accuracy: 0.8125, lr: 0.0004
batch: 150, loss: 0.4743, accuracy: 0.8750, lr: 0.0004
batch: 160, loss: 0.4569, accuracy: 1.0000, lr: 0.0004
batch: 170, loss: 0.5653, accuracy: 0.7500, lr: 0.0004
batch: 180, loss: 0.4

## 测试

In [72]:
def test(model):
    # 定义测试的数据加载器
    data_loader_test = torch.utils.data.DataLoader(
        dataset=Dataset('test'),
        batch_size=32,
        collate_fn=collate_fn,
        shuffle=True,
        drop_last=True
    )
    # 切换到测试模式
    model.eval()

    correct = 0
    total = 0

    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(data_loader_test):
        if i == 5:
            break

        # 计算
        with torch.no_grad():
            if device != 'cpu':
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                token_type_ids = token_type_ids.to(device)
                labels = labels.to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        out = out.argmax(dim=1)
        correct += (out == labels).sum().item()
        total += len(labels)
    print(f'accuracy: {correct / total:.4f}')

In [73]:
test(model)

accuracy: 0.8812
